In [1]:
import torch
import torch.nn as nn

class PositionalEncoding(nn.Module):
    def __init__(self, max_length, embed_dim, dropout=0.1):
        super().__init__()
        self.pos_embed = nn.Parameter(torch.randn(max_length, embed_dim) * 0.02)
        self.dropout = nn.Dropout(dropout)

    def forward(self, X):
        return self.dropout(X + self.pos_embed[:X.size(1)])

In [2]:
class FixedPositionalEncoding(nn.Module):
    def __init__(self, max_length, embed_dim, dropout=0.1):
        super().__init__()
        p = torch.arange(max_length).unsqueeze(1)
        i = torch.arange(0, embed_dim, 2)
        angle = p / 10_000 ** (i / embed_dim)
        pos_encodings = torch.empty(max_length, embed_dim)
        pos_encodings[:, ::2] = angle.sin()
        pos_encodings[:, 1::2] = angle.cos()
        self.register_buffer("pos_encodings", pos_encodings)
        self.dropout = nn.Dropout(dropout)

    def forward(self, X):
        return self.dropout(X + self.pos_encodings[:X.size(1)])

In [3]:
class MultiHeadAttention(nn.Module):
    def __init__(self, embed_dim, num_heads, dropout=0.1):
        super().__init__()
        self.h = num_heads
        self.d = embed_dim // num_heads
        self.q_proj = nn.Linear(embed_dim, embed_dim)
        self.k_proj = nn.Linear(embed_dim, embed_dim)
        self.v_proj = nn.Linear(embed_dim, embed_dim)
        self.out_proj = nn.Linear(embed_dim, embed_dim)
        self.dropout = nn.Dropout(dropout)

    def split_heads(self, X):
        return X.view(X.size(0), X.size(1), self.h, self.d).transpose(1, 2)

    def forward(self, query, key, value, attn_mask=None, key_padding_mask=None):
        q = self.split_heads(self.q_proj(query))  # (B, h, Lq, d)
        k = self.split_heads(self.k_proj(key))  # (B, h, Lk, d)
        v = self.split_heads(self.v_proj(value))  # (B, h, Lv, d) with Lv=Lk
        scores = q @ k.transpose(2, 3) / self.d**0.5  # (B, h, Lq, Lk)

        #Masking
        if attn_mask is not None:
            scores = scores.masked_fill(attn_mask, -torch.inf)
        if key_padding_mask is not None:
            mask = key_padding_mask.unsqueeze(1).unsqueeze(2)
            scores = scores.masked_fill(mask, -torch.inf)

        weights = scores.softmax(dim=-1)  # (B, h, Lq, Lk)
        Z = self.dropout(weights) @ v  # (B, h, Lq, d)
        Z = Z.transpose(1, 2)  # (B, Lq, h, d)
        Z = Z.reshape(Z.size(0), Z.size(1), self.h * self.d)  # (B, Lq, h × d)
        return (self.out_proj(Z), weights)  # (B, Lq, h × d)

# Alternative from PyTorch, same purpose as above
# nn.MultiheadAttention(embedding_dim, num_heads, dropout, batch_first=True)

In [4]:
class TransformerEncoderLayer(nn.Module):
    def __init__(self, d_model, n_head, dim_feedforward=2048, dropout=0.1):
        super().__init__()
        self.self_attn = MultiHeadAttention(d_model, n_head, dropout)
        self.linear1 = nn.Linear(d_model, dim_feedforward)
        self.dropout = nn.Dropout(dropout)
        self.linear2 = nn.Linear(dim_feedforward, d_model)
        self.norm1 = nn.LayerNorm(d_model)
        self.norm2 = nn.LayerNorm(d_model)

    def forward(self, src, src_mask=None, src_key_padding_mask=None):
        attn, _ = self.self_attn(src, src, src,
                                 src_mask, src_key_padding_mask)
        Z = self.norm1(src + self.dropout(attn))
        ff = self.dropout(self.linear2(self.dropout(self.linear1(Z).relu())))
        return self.norm2(Z + ff)

In [5]:
class TransformerDecoderLayer(nn.Module):
    def __init__(self, d_model, n_head, dim_feedforward=2048, dropout=0.1):
        super().__init__()
        self.self_attn = MultiHeadAttention(d_model, n_head, dropout)
        self.multihead_attn = MultiHeadAttention(d_model, n_head, dropout)
        self.dropout = nn.Dropout(dropout)
        self.linear1 = nn.Linear(d_model, dim_feedforward)
        self.linear2 = nn.Linear(dim_feedforward, d_model)
        self.norm1 = nn.LayerNorm(d_model)
        self.norm2 = nn.LayerNorm(d_model)
        self.norm3 = nn.LayerNorm(d_model)


    def forward(self, tgt, memory, tgt_mask=None, memory_mask=None,
                tgt_key_padding_mask=None, memory_key_padding_mask=None):
        attn1, _ = self.self_attn(tgt, tgt, tgt, attn_mask=tgt_mask,
                                  key_padding_mask=tgt_key_padding_mask)
        Z = self.norm1(tgt + self.dropout(attn1))
        attn2, _ = self.multihead_attn(Z, memory, memory, attn_mask=memory_mask,
                                       key_padding_mask=memory_key_padding_mask)
        Z = self.norm2(Z + self.dropout(attn2))
        ff = self.dropout(self.linear2(self.dropout(self.linear1(Z).relu())))
        return self.norm3(Z + ff)

In [7]:
#Alternative from PyTorch, same purpose as above

d_model = 512
n_head = 8
dim_feedforward = 2048
dropout = 0.1
num_encoder_layers = 6
num_decoder_layers = 6

encoder_layer = nn.TransformerEncoderLayer(d_model, n_head, dim_feedforward, dropout, batch_first=True)
encoder = nn.TransformerEncoder(encoder_layer, num_layers=num_encoder_layers)

decoder_layer = nn.TransformerDecoderLayer(d_model, n_head, dim_feedforward, dropout, batch_first=True)
decoder = nn.TransformerDecoder(decoder_layer, num_layers=num_decoder_layers)

model = nn.Transformer(
    d_model=d_model,
    nhead=n_head,
    num_encoder_layers=num_encoder_layers,
    num_decoder_layers=num_decoder_layers,
    dim_feedforward=dim_feedforward,
    dropout=dropout,
    batch_first=True,
)

In [12]:
class NmtTransformer(nn.Module):
    def __init__(self, vocab_size, max_length, embed_dim=512, pad_id=0,
                 num_heads=8, num_layers=6, dropout=0.1):
        super().__init__()
        self.embed = nn.Embedding(vocab_size, embed_dim, padding_idx=pad_id)
        self.pos_embed = PositionalEncoding(max_length, embed_dim, dropout)
        self.transformer = nn.Transformer(
            embed_dim, num_heads, num_encoder_layers=num_layers,
            num_decoder_layers=num_layers, batch_first=True)
        self.output = nn.Linear(embed_dim, vocab_size)

    def forward(self, pair): #Nmt_Pair class
        src_embeds = self.pos_embed(self.embed(pair.src_token_ids))
        tgt_embeds = self.pos_embed(self.embed(pair.tgt_token_ids))
        src_pad_mask = ~pair.src_mask.bool()
        tgt_pad_mask = ~pair.tgt_mask.bool()
        size = [pair.tgt_token_ids.size(1)] * 2
        full_mask = torch.full(size, True, device=tgt_pad_mask.device)
        causal_mask = torch.triu(full_mask, diagonal=1)
        out_decoder = self.transformer(src_embeds, tgt_embeds,
                                       src_key_padding_mask=src_pad_mask,
                                       memory_key_padding_mask=src_pad_mask,
                                       tgt_mask=causal_mask, tgt_is_causal=True,
                                       tgt_key_padding_mask=tgt_pad_mask)
        return self.output(out_decoder).permute(0, 2, 1)

In [13]:
import torch.nn.functional as F
from torch.utils.data import DataLoader
from collections import namedtuple
import tokenizers
from datasets import load_dataset
import torchmetrics

device = "cuda" if torch.cuda.is_available() else "cpu"

# ---- Dataset ----
nmt_valid_full, nmt_test_set = load_dataset(
    path="ageron/tatoeba_mt_train", name="eng-spa",
    split=["validation", "test"])
split = nmt_valid_full.train_test_split(train_size=0.8, seed=42)
nmt_train_set, nmt_valid_set = split["train"], split["test"]

# ---- Tokenizer ----
max_length = 256
vocab_size = 10000

def train_eng_spa():
    for pair in nmt_train_set:
        yield pair["source_text"]
        yield pair["target_text"]

nmt_tokenizer = tokenizers.Tokenizer(tokenizers.models.BPE(unk_token="<unk>"))
nmt_tokenizer.enable_padding(pad_id=0, pad_token="<pad>")
nmt_tokenizer.enable_truncation(max_length=max_length)
nmt_tokenizer.pre_tokenizer = tokenizers.pre_tokenizers.Whitespace()
trainer = tokenizers.trainers.BpeTrainer(
    vocab_size=vocab_size, special_tokens=["<pad>", "<unk>", "<s>", "</s>"])
nmt_tokenizer.train_from_iterator(train_eng_spa(), trainer)
vocab_size = nmt_tokenizer.get_vocab_size()

# ---- NmtPair + collate_fn ----
fields = ["src_token_ids", "src_mask", "tgt_token_ids", "tgt_mask"]
class NmtPair(namedtuple("NmtPairBase", fields)):
    def to(self, device, **kwargs):
        return NmtPair(*[t.to(device, **kwargs) for t in self])

def nmt_collate_fn(batch):
    src_texts = [pair["source_text"] for pair in batch]
    tgt_texts = [f"<s> {pair['target_text']} </s>" for pair in batch]

    src_enc = nmt_tokenizer.encode_batch(src_texts)
    tgt_enc = nmt_tokenizer.encode_batch(tgt_texts)

    src_ids = torch.tensor([e.ids for e in src_enc])
    tgt_ids = torch.tensor([e.ids for e in tgt_enc])
    src_mask = torch.tensor([e.attention_mask for e in src_enc])
    tgt_mask = torch.tensor([e.attention_mask for e in tgt_enc])

    inputs = NmtPair(src_ids, src_mask, tgt_ids[:, :-1], tgt_mask[:, :-1])
    labels = tgt_ids[:, 1:]
    return inputs, labels

batch_size = 32
nmt_train_loader = DataLoader(nmt_train_set, batch_size=batch_size,
                              collate_fn=nmt_collate_fn, shuffle=True)
nmt_valid_loader = DataLoader(nmt_valid_set, batch_size=batch_size,
                              collate_fn=nmt_collate_fn)



torch.manual_seed(42)
nmt_tr_model = NmtTransformer(vocab_size, max_length, embed_dim=128, pad_id=0,
                              num_heads=4, num_layers=2, dropout=0.1).to(device)

# ---- Training loop (compact, no external util) ----
xentropy = nn.CrossEntropyLoss(ignore_index=0)
optimizer = torch.optim.NAdam(nmt_tr_model.parameters(), lr=1e-3)
metric = torchmetrics.Accuracy(task="multiclass", num_classes=vocab_size,
                                ignore_index=0).to(device)

def run_epoch(model, loader, train_mode, optimizer=None):
    model.train() if train_mode else model.eval()
    metric.reset()
    total_loss = 0
    with torch.set_grad_enabled(train_mode):
        for inputs, labels in loader:
            inputs, labels = inputs.to(device), labels.to(device)
            logits = model(inputs)
            loss = xentropy(logits, labels)
            if train_mode:
                optimizer.zero_grad()
                loss.backward()
                optimizer.step()
            total_loss += loss.item()
            metric.update(logits, labels)
    return total_loss / len(loader), metric.compute().item()

num_epochs = 20
for epoch in range(num_epochs):
    train_loss, train_acc = run_epoch(nmt_tr_model, nmt_train_loader, True, optimizer)
    valid_loss, valid_acc = run_epoch(nmt_tr_model, nmt_valid_loader, False)
    print(f"Epoch {epoch+1}/{num_epochs} — "
          f"train loss: {train_loss:.4f}, acc: {train_acc:.4f} | "
          f"valid loss: {valid_loss:.4f}, acc: {valid_acc:.4f}")

/home/denys/PycharmProjects/Hands-On_ML/.venv/lib/python3.12/site-packages/torch/nn/modules/transformer.py:508: UserWarning: The PyTorch API of nested tensors is in prototype stage and will change in the near future. We recommend specifying layout=torch.jagged when constructing a nested tensor, as this layout receives active development, has better operator coverage, and works with torch.compile. (Triggered internally at /pytorch/aten/src/ATen/NestedTensorImpl.cpp:178.)
  output = torch._nested_tensor_from_mask(


Epoch 1/20 — train loss: 3.6756, acc: 0.4177 | valid loss: 2.6116, acc: 0.5388
Epoch 2/20 — train loss: 2.5270, acc: 0.5385 | valid loss: 2.1814, acc: 0.5926
Epoch 3/20 — train loss: 2.2084, acc: 0.5770 | valid loss: 1.9934, acc: 0.6222
Epoch 4/20 — train loss: 2.0395, acc: 0.5984 | valid loss: 1.8768, acc: 0.6377
Epoch 5/20 — train loss: 1.9269, acc: 0.6132 | valid loss: 1.8053, acc: 0.6496
Epoch 6/20 — train loss: 1.8419, acc: 0.6247 | valid loss: 1.7637, acc: 0.6569


KeyboardInterrupt: 

In [14]:
def translate(model, src_text, max_length=20, pad_id=0, eos_id=3):
    tgt_text = ""
    token_ids = []
    for index in range(max_length):
        batch, _ = nmt_collate_fn([{"source_text": src_text,
                                    "target_text": tgt_text}])
        with torch.no_grad():
            Y_logits = model(batch.to(device))
            Y_token_ids = Y_logits.argmax(dim=1)  # find the best token IDs
            next_token_id = Y_token_ids[0, index]  # take the last token ID

        next_token = nmt_tokenizer.id_to_token(next_token_id)
        tgt_text += " " + next_token
        if next_token_id == eos_id:
            break
    return tgt_text

nmt_tr_model.eval()
translate(nmt_tr_model,"I like to play soccer with my friends at the beach")

' Me gusta jugar fútbol con mis amigos en la playa . </s>'

In [ ]:
class MultiheadAttentionWithRPB(nn.Module):
    def __init__(self, embed_dim, num_heads, max_distance=128, dropout=0.1):
        super().__init__()
        self.h = num_heads
        self.d = embed_dim // num_heads
        self.q_proj = nn.Linear(embed_dim, embed_dim)
        self.k_proj = nn.Linear(embed_dim, embed_dim)
        self.v_proj = nn.Linear(embed_dim, embed_dim)
        self.out_proj = nn.Linear(embed_dim, embed_dim)
        self.dropout = nn.Dropout(dropout)

        self.bias_table = nn.Embedding(2 * max_distance + 1, num_heads)
        self.max_distance = max_distance

    def split_heads(self, X):
        return X.view(X.size(0), X.size(1), self.h, self.d).transpose(1, 2)

    def forward(self, query, key, value):
        q = self.split_heads(self.q_proj(query))
        k = self.split_heads(self.k_proj(key))
        v = self.split_heads(self.v_proj(value))

        scores = q @ k.transpose(2, 3) / self.d**0.5

        #RPB modification
        Lq, Lk = q.size(2), k.size(2)
        q_pos = torch.arange(Lq, device=q.device)[:, None]
        k_pos = torch.arange(Lk, device=q.device)[None, :]
        rel_pos = (k_pos - q_pos).clamp(-self.max_distance, self.max_distance)
        idx = rel_pos + self.max_distance
        bias = self.bias_table(idx).permute(2, 0, 1)
        scores = scores + bias

        weights = scores.softmax(dim=-1)
        Z = self.dropout(weights) @ v
        Z = Z.transpose(1, 2).reshape(Z.size(0), Z.size(1), self.h * self.d)
        return self.out_proj(Z), weights

In [1]:
from transformers import BertConfig, BertForMaskedLM, BertTokenizer

bert_tokenizer = BertTokenizer.from_pretrained("bert-base-uncased")
config = BertConfig(
    vocab_size=bert_tokenizer.vocab_size, hidden_size=128, num_hidden_layers=2,
    num_attention_heads=4, intermediate_size=512, max_position_embeddings=128)
bert = BertForMaskedLM(config)

In [3]:
from datasets import load_dataset

def tokenize(example, tokenizer=bert_tokenizer):
    return tokenizer(example["text"], truncation=True, max_length=128, padding="max_length")

mlm_dataset = load_dataset("Salesforce/wikitext", "wikitext-2-raw-v1", split="train")
mlm_dataset = mlm_dataset.map(tokenize, batched=True)

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

wikitext-2-raw-v1/test-00000-of-00001.pa(…):   0%|          | 0.00/733k [00:00<?, ?B/s]

wikitext-2-raw-v1/train-00000-of-00001.p(…):   0%|          | 0.00/6.36M [00:00<?, ?B/s]

wikitext-2-raw-v1/validation-00000-of-00(…):   0%|          | 0.00/657k [00:00<?, ?B/s]

Generating test split:   0%|          | 0/4358 [00:00<?, ? examples/s]

Generating train split:   0%|          | 0/36718 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/3760 [00:00<?, ? examples/s]

Map:   0%|          | 0/36718 [00:00<?, ? examples/s]

In [8]:
from transformers import Trainer, TrainingArguments
from transformers import DataCollatorForLanguageModeling

args = TrainingArguments(output_dir="./my_bert", num_train_epochs=5, per_device_train_batch_size=64)

mlm_collator = DataCollatorForLanguageModeling(tokenizer=bert_tokenizer, mlm=True, mlm_probability=0.15)

trainer = Trainer(model=bert, args=args, train_dataset=mlm_dataset, data_collator=mlm_collator)

trainer_output = trainer.train()

Step,Training Loss
500,7.045293
1000,7.042321
1500,6.984174
2000,6.957389
2500,6.945105


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

In [11]:
#Definitely the comma is not the result expected. Longer training is required and on larger dataset

import torch
from transformers import pipeline
torch.manual_seed(42)
fill_mask = pipeline("fill-mask", model=bert, tokenizer=bert_tokenizer)
top_predictions = fill_mask("The capital of [MASK] is Rome")
top_predictions[0]

{'score': 0.04949595779180527,
 'token': 1010,
 'token_str': ',',
 'sequence': 'the capital of, is rome'}